# 01 -- Data Cleaning & Merging

Cilem tohoto notebooku je:
1. Nacist a vycistit cenova data akcii (yfinance)
2. Nacist a vycistit Reddit sentiment data (historicky Kaggle dataset)
3. Otagovat Reddit posty podle zminovanych tickeru
4. Agregovat pocet zminek na denni urovni per ticker
5. Ulozit vycistena data do `data/processed/` pro dalsi analyzu

**Dulezita poznamka k metodologii:** V tomto notebooku zatim neresime spojeni
cen a sentimentu do jedne casove osy -- to delame az v dalsim notebooku,
kde budeme muset dat extra pozor na *look-ahead bias* (sentiment k danemu
dni musi byt dostupny pred cenovym pohybem, ktery se snazime vysvetlit,
ne po nem).

## Import knihoven

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Cesty k datum

Notebook lezi v `notebooks/`, takze vsechny cesty jdou o uroven vys.

In [ ]:
PROJECT_ROOT = Path.cwd().parent
PRICES_DIR = PROJECT_ROOT / "data" / "raw" / "prices"
REDDIT_DIR = PROJECT_ROOT / "data" / "raw" / "reddit"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Prices dir:  {PRICES_DIR}")
print(f"Reddit dir:  {REDDIT_DIR}")
print(f"Processed:   {PROCESSED_DIR}")

## 1. Nacteni cenovych dat

Kazdy ticker ma svuj CSV v `data/raw/prices/`. Nacteme vsechny a spojime
do jednoho DataFrame se sloupcem `Ticker`.

In [ ]:
price_files = sorted(PRICES_DIR.glob("*.csv"))
print(f"Nalezeno {len(price_files)} souboru:")
for f in price_files:
    print(f"  {f.name}")

In [ ]:
price_dfs = []

for f in price_files:
    df = pd.read_csv(f)
    price_dfs.append(df)

prices = pd.concat(price_dfs, ignore_index=True)
print(f"Celkem radku: {len(prices)}")
prices.head()

### Cisteni cenovych dat

Par veci, ktere je potreba osetrit:
- sloupec `Date` je zatim text, prevedeme na datetime
- zkontrolujeme chybejici hodnoty (yfinance obcas vraci NaN u objemu/ceny
  pro dny, kdy se s akcii neobchodovalo, napr. svatky)
- serad'ime podle tickeru a data

In [ ]:
prices["Date"] = pd.to_datetime(prices["Date"])

print("Chybejici hodnoty po sloupcich:")
print(prices.isna().sum())

In [ ]:
# Pokud existuji radky s chybejicimi cenami, podivame se na ne blize
# nez se rozhodneme, jak je osetrit (smazat vs. dopocitat)
missing_rows = prices[prices.isna().any(axis=1)]
missing_rows

In [ ]:
# Pokud jsou chybejici radky jen ojedinele (napr. svatky), je bezpecne
# je odstranit -- pro analyzu sentiment vs. cena nam chybejici den nevadi,
# proste pro nej nebudeme mit signal
prices = prices.dropna()

prices = prices.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"Po cisteni: {len(prices)} radku")
prices.head()

In [ ]:
# Rychla kontrola casoveho rozpeti per ticker -- pripomenme si, ze PLTR
# ma kratsi historii kvuli pozdejsimu IPO (zari 2020)
prices.groupby("Ticker")["Date"].agg(["min", "max", "count"])

### Vypocet dennich vynosu a volatility

Pro pozdejsi backtest budeme potrebovat denni procentualni zmenu ceny
(return) a klouzavou volatilitu -- pridame je uz ted, at je mame hotove.

In [ ]:
prices["Daily_Return"] = prices.groupby("Ticker")["Close"].pct_change()

# 5-denni klouzava volatilita (smerodatna odchylka vynosu) jako jednoducha
# proxy pro "jak moc se dana akcie prave hejbe"
prices["Volatility_5d"] = (
    prices.groupby("Ticker")["Daily_Return"]
    .transform(lambda x: x.rolling(window=5).std())
)

prices.head(10)

## 2. Nacteni Reddit dat

Nacteme historicky dataset a rovnou opravime dva problemy, na ktere jsme
narazili pri inspekci:
1. `created_utc` je Unix timestamp v sekundach (ne nanosekundach)
2. matching tickeru musi byt na cela slova, ne podretezce (jinak napr.
   "KO" chytne cast slova "looking")

In [ ]:
reddit_files = list(REDDIT_DIR.glob("*.csv"))
assert len(reddit_files) >= 1, "Nenalezen zadny CSV v data/raw/reddit/"
reddit_path = reddit_files[0]
print(f"Nacitam: {reddit_path.name}")

reddit = pd.read_csv(reddit_path, low_memory=False)
print(f"Celkem radku: {len(reddit)}")
reddit.head()

In [ ]:
reddit["created_utc"] = pd.to_datetime(reddit["created_utc"], unit="s", utc=True)

# Pro spojeni s cenovymi daty (ktera jsou po dnech, bez casove zony)
# potrebujeme jen datum, ne presny cas
reddit["Date"] = reddit["created_utc"].dt.tz_localize(None).dt.normalize()

print(f"Casove rozpeti: {reddit['Date'].min()} az {reddit['Date'].max()}")

### Oriznuti na relevantni obdobi

Cenova data mame od 2020-01-01, takze Reddit data z let 2012-2019 nam
k nicemu nebudou -- oriznuti hned na zacatku setri pamet i cas pri
dalsim zpracovani.

In [ ]:
PRICE_START = prices["Date"].min()
print(f"Cenova data zacinaji: {PRICE_START}")

reddit = reddit[reddit["Date"] >= PRICE_START].reset_index(drop=True)
print(f"Reddit radku po oriznuti: {len(reddit)}")

### Tagovani postu podle tickeru

Pouzivame stejny pristup jako v inspekcnim skriptu -- hledame cele slovo
(alias nazvu firmy nebo symbol tickeru), ne podretezec. Jeden post muze
zminovat vic tickeru najednou, takze vysledek bude "dlouhy" format:
jeden radek = (post, ticker), ne (post).

In [ ]:
TICKER_ALIASES = {
    "AAPL": ["AAPL", "Apple"],
    "TSLA": ["TSLA", "Tesla"],
    "NVDA": ["NVDA", "Nvidia"],
    "MSFT": ["MSFT", "Microsoft"],
    "AMZN": ["AMZN", "Amazon"],
    "GME":  ["GME", "Gamestop", "GameStop"],
    "AMC":  ["AMC"],
    "PLTR": ["PLTR", "Palantir"],
    "DIS":  ["DIS", "Disney"],
}

# Predkompilovane regexy pro rychlost -- vsechny aliasy pro dany ticker
# spojime do jednoho patternu pomoci OR (|)
ticker_patterns = {
    ticker: re.compile(
        r"\b(" + "|".join(re.escape(a.lower()) for a in aliases) + r")\b"
    )
    for ticker, aliases in TICKER_ALIASES.items()
}

In [ ]:
reddit["title_lower"] = reddit["title"].astype(str).str.lower()

# Pro kazdy ticker vytvorime bool sloupec, jestli se v postu zminuje
for ticker, pattern in ticker_patterns.items():
    reddit[f"mentions_{ticker}"] = reddit["title_lower"].str.contains(pattern, na=False)

mention_cols = [f"mentions_{t}" for t in TICKER_ALIASES]
reddit["any_mention"] = reddit[mention_cols].any(axis=1)

print(f"Postu s alespon jednou relevantni zminkou: {reddit['any_mention'].sum()} z {len(reddit)}")

In [ ]:
# Prevod na "long" format: jeden radek = (post, ticker)
# tohle nam usnadni pozdejsi agregaci per ticker per den
relevant = reddit[reddit["any_mention"]].copy()

long_rows = []
for ticker in TICKER_ALIASES:
    subset = relevant[relevant[f"mentions_{ticker}"]][["id", "Date", "title", "score", "num_comments"]].copy()
    subset["Ticker"] = ticker
    long_rows.append(subset)

reddit_long = pd.concat(long_rows, ignore_index=True)
print(f"Celkem (post, ticker) paru: {len(reddit_long)}")
reddit_long.head()

### Agregace na denni urovni

Pro kazdy ticker a den spocitame:
- pocet postu (`post_count`) -- proxy pro "objem pozornosti"
- soucet skore a komentaru -- proxy pro "angazovanost" komunity

Skutecny sentiment (pozitivni/negativni) spocitame az v dalsim notebooku
pomoci FinBERT -- tady zatim jen pripravujeme strukturu dat.

In [ ]:
daily_mentions = (
    reddit_long
    .groupby(["Ticker", "Date"])
    .agg(
        post_count=("id", "count"),
        total_score=("score", "sum"),
        total_comments=("num_comments", "sum"),
    )
    .reset_index()
    .sort_values(["Ticker", "Date"])
)

daily_mentions.head(10)

In [ ]:
# Rychla kontrola pokryti -- kolik dni ma kazdy ticker alespon 1 zminku
daily_mentions.groupby("Ticker")["Date"].count().sort_values(ascending=False)

## 3. Ulozeni vycistenych dat

Ukladame dve samostatne tabulky do `data/processed/`:
- `prices_clean.csv` -- denni ceny + returns + volatilita
- `reddit_daily_mentions.csv` -- denni pocty zminek per ticker

Spojeni obou do jedne casove osy (a reseni look-ahead bias) budeme delat
az v dalsim notebooku, protoze je to samostatny, dulezity krok, ktery
si zaslouzi vlastni prostor a peclivou kontrolu.

In [ ]:
prices.to_csv(PROCESSED_DIR / "prices_clean.csv", index=False)
daily_mentions.to_csv(PROCESSED_DIR / "reddit_daily_mentions.csv", index=False)

# Ukladame i post-level data s textem -- budeme je potrebovat v notebooku 02
# pro FinBERT sentiment scoring (tam uz nestaci jen agregovane pocty,
# potrebujeme skutecny text kazdeho postu)
reddit_long.to_csv(PROCESSED_DIR / "reddit_posts_tagged.csv", index=False)

print("Ulozeno:")
print(f"  {PROCESSED_DIR / 'prices_clean.csv'}  ({len(prices)} radku)")
print(f"  {PROCESSED_DIR / 'reddit_daily_mentions.csv'}  ({len(daily_mentions)} radku)")
print(f"  {PROCESSED_DIR / 'reddit_posts_tagged.csv'}  ({len(reddit_long)} radku)")

## Shrnuti

- Cenova data: {n_tickers} tickeru, ocistena, s dopocitanymi returns a volatilitou
- Reddit data: orizuta na relevantni obdobi, otagovana podle tickeru, agregovana na denni urovni
- Obe tabulky ulozeny do `data/processed/` a pripravene na spojeni

**Dalsi krok (notebook 02):** spojeni obou datasetu na spolecnou casovou
osu s dukladnym osetrenim look-ahead bias, a prvni pohled na to, jestli
mezi vykyvy v poctu zminek a nasledujicimi cenovymi pohyby vubec neco je.